# Polars intro

Polars is a blazingly fast DataFrame library for manipulating structured data. The core is written in Rust, and available for Python, R and NodeJS.

### Key features

- **Fast**: Written from scratch in Rust, designed close to the machine and without external
  dependencies.
- **I/O**: First class support for all common data storage layers: local, cloud storage & databases.
- **Intuitive API**: Write your queries the way they were intended. Polars, internally, will
  determine the most efficient way to execute using its query optimizer.
- **Out of Core**: The streaming API allows you to process your results without requiring all your
  data to be in memory at the same time.
- **Parallel**: Utilises the power of your machine by dividing the workload among the available CPU cores without any additional configuration.
- **Vectorized Query Engine**
- **GPU Support**: Optionally run queries on NVIDIA GPUs for maximum performance for in-memory workloads.
- **[Apache Arrow support](https://arrow.apache.org/)**: Polars can consume and produce Arrow data
  often with zero-copy operations. Note that Polars is not built on a Pyarrow/Arrow implementation.
  Instead, Polars has its own compute and buffer implementations.

### Potential annoyances for xarray users
- **2D DataFrames only**. No N-D data, but ways to still do nD things with `.group_by`
- **Very different syntax** from `numpy` and `pandas`

### Basically a mix of `pandas`, `SQL` and `dask` 
but with its own syntax. 

In [58]:
import polars as pl
import datetime as dt

df = pl.read_csv("/Users/bandelol/Documents/code_local/data/iris.csv") # I/O like pandas (read and write, not open and to)
df.head()


sepal_length,sepal_width,petal_length,petal_width,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""Setosa"""
4.9,3.0,1.4,0.2,"""Setosa"""
4.7,3.2,1.3,0.2,"""Setosa"""
4.6,3.1,1.5,0.2,"""Setosa"""
5.0,3.6,1.4,0.2,"""Setosa"""


In [23]:
df["species"].unique()

species
str
"""Virginica"""
"""Versicolor"""
"""Setosa"""


In [ ]:
pl.lit([2, 4])

In [57]:
expr = ((pl.col("sepal_length") + 32) * 3920).sqrt().pow(2)

In [60]:
df.with_columns(custum_col=expr)

sepal_length,sepal_width,petal_length,petal_width,species,custum_col
f64,f64,f64,f64,str,f64
5.1,3.5,1.4,0.2,"""Setosa""",145432.0
4.9,3.0,1.4,0.2,"""Setosa""",144648.0
4.7,3.2,1.3,0.2,"""Setosa""",143864.0
4.6,3.1,1.5,0.2,"""Setosa""",143472.0
5.0,3.6,1.4,0.2,"""Setosa""",145040.0
…,…,…,…,…,…
6.7,3.0,5.2,2.3,"""Virginica""",151704.0
6.3,2.5,5.0,1.9,"""Virginica""",150136.0
6.5,3.0,5.2,2.0,"""Virginica""",150920.0


In [47]:
q = (
    df
    .filter(pl.col("sepal_length") > 5) # filter using an Expression (equivalent-ish of boolean indexing)
    .group_by(pl.col("species"), maintain_order=True) # group by, super useful
    .agg(pl.all().mean()) # group wise aggregation, also using an Expression
)
q

species,sepal_length,sepal_width,petal_length,petal_width
str,f64,f64,f64,f64
"""Setosa""",5.313636,3.713636,1.509091,0.277273
"""Versicolor""",5.997872,2.804255,4.317021,1.346809
"""Virginica""",6.622449,2.983673,5.573469,2.032653


In [4]:
import polars as pl
import datetime as dt

df = pl.DataFrame(
    {
        "name": ["Alice Archer", "Ben Brown", "Chloe Cooper", "Daniel Donovan"],
        "birthdate": [
            dt.date(1997, 1, 10),
            dt.date(1985, 2, 15),
            dt.date(1983, 3, 22),
            dt.date(1981, 4, 30),
        ],
        "weight": [57.9, 72.5, 53.6, 83.1],  # (kg)
        "height": [1.56, 1.77, 1.65, 1.75],  # (m)
    }
)

df

name,birthdate,weight,height
str,date,f64,f64
"""Alice Archer""",1997-01-10,57.9,1.56
"""Ben Brown""",1985-02-15,72.5,1.77
"""Chloe Cooper""",1983-03-22,53.6,1.65
"""Daniel Donovan""",1981-04-30,83.1,1.75


# tricks for xarray users

In [61]:
import xarray as xr
da = xr.open_mfdataset("/Users/bandelol/Documents/code_local/data/t2m/199?.nc")["t"]
da = da.sel(lon=slice(-80, 40), lat=slice(15, 80))

In [64]:
da = da.compute()
da

<xarray.DataArray 't' (time: 1826, lat: 131, lon: 241)> Size: 231MB
array([[[-3.44543457e-02, -9.01489258e-02, -7.04956055e-02, ...,
          5.50872803e-01,  1.66992188e-01,  7.27264404e-01],
        [-6.45446777e-02, -9.73510742e-03,  6.18591309e-02, ...,
          7.71545410e-01,  3.95629883e-01,  1.92779541e-01],
        [ 9.01794434e-02,  1.08306885e-01,  1.27288818e-01, ...,
          9.25384521e-01,  2.86834717e-01,  2.71484375e-01],
        ...,
        [ 1.61648560e+00,  1.52163696e+00,  1.75579834e+00, ...,
         -3.66891479e+00, -3.74063110e+00, -3.84983826e+00],
        [ 3.58119202e+00,  3.83929443e+00,  3.96601868e+00, ...,
         -2.77854919e+00, -2.93864441e+00, -3.09771729e+00],
        [ 3.15974426e+00,  3.33349609e+00,  3.87300110e+00, ...,
         -2.52650452e+00, -2.51338196e+00, -2.54832458e+00]],

       [[-2.76885986e-01, -2.68310547e-01, -1.76300049e-01, ...,
          5.16052246e-01,  1.69097900e-01,  1.62078857e-01],
        [-1.56311035e-01, -2.44750977e-01, -2.48504639e-01, ...,
          4.85351562e-01,  2.80792236e-01, -2.67333984e-02],
        [ 1.84020996e-02, -9.35974121e-02, -2.72247314e-01, ...,
          4.16687012e-01,  2.49389648e-01,  2.87689209e-01],
...
         -3.73013306e+00, -3.73947144e+00, -3.76374817e+00],
        [-2.62823486e+00, -9.50256348e-01, -3.47900391e-03, ...,
         -2.93257141e+00, -2.97633362e+00, -3.01681519e+00],
        [ 6.86859131e-01,  5.63751221e-01,  1.26068115e+00, ...,
         -2.00474548e+00, -2.06904602e+00, -2.12403870e+00]],

       [[-2.88116455e-01, -8.09020996e-02, -5.67932129e-02, ...,
         -5.64880371e-02, -3.60107422e-01, -2.69653320e-01],
        [-4.02130127e-01, -1.20452881e-01, -5.98144531e-03, ...,
         -8.44726562e-02,  6.53076172e-03, -3.04412842e-01],
        [-4.21203613e-01, -1.63238525e-01, -5.58471680e-03, ...,
          5.34881592e-01,  2.91137695e-02, -3.54064941e-01],
        ...,
        [ 2.89311218e+00,  2.74519348e+00,  2.23107910e+00, ...,
         -3.79476929e+00, -3.87692261e+00, -3.96620178e+00],
        [ 5.64666748e-01,  1.04847717e+00,  1.30851746e+00, ...,
         -2.64151001e+00, -2.73188782e+00, -2.80625916e+00],
        [ 2.46434021e+00,  2.16098022e+00,  2.84065247e+00, ...,
         -2.45774841e+00, -2.32102966e+00, -2.26219177e+00]]],
      dtype=float32)
Coordinates:
  * lat      (lat) float32 524B 15.0 15.5 16.0 16.5 17.0 ... 78.5 79.0 79.5 80.0
  * lon      (lon) float32 964B -80.0 -79.5 -79.0 -78.5 ... 38.5 39.0 39.5 40.0
  * time     (time) datetime64[ns] 15kB 1990-01-01 1990-01-02 ... 1994-12-31

In [48]:
gb = da.groupby("time.dayofyear")
clim = gb.mean().compute()
anom = gb - clim
anom.compute()

<xarray.DataArray 't' (time: 1826, lat: 131, lon: 241)> Size: 231MB
array([[[ 0.0975647 ,  0.04778442,  0.07958373, ..., -0.14196777,
         -0.594281  , -0.18191528],
        [ 0.0755188 ,  0.07476807,  0.13668212, ..., -0.13345337,
         -0.12252808, -0.21696168],
        [ 0.23527832,  0.20772704,  0.17626342, ...,  0.32437742,
         -0.10099488, -0.05635986],
        ...,
        [ 0.5159546 ,  0.5559753 ,  0.57758486, ..., -0.915503  ,
         -0.9742615 , -1.0999389 ],
        [ 1.7558472 ,  2.0642457 ,  2.2317688 , ..., -0.31897283,
         -0.5089936 , -0.6933868 ],
        [ 1.1011047 ,  1.5169281 ,  1.9958527 , ..., -0.49009085,
         -0.43858647, -0.4504609 ]],

       [[ 0.04002076,  0.0180847 ,  0.14217529, ...,  0.3128662 ,
         -0.04168701, -0.11603394],
        [ 0.13281861,  0.03807372,  0.0595459 , ...,  0.13570556,
          0.21346435, -0.03211059],
        [ 0.30984497,  0.20451659,  0.06712037, ...,  0.37421265,
          0.2581543 ,  0.19885254],
...
        [-1.1508546 , -1.0688844 , -1.596112  , ..., -1.7774231 ,
         -1.6532044 , -1.5341096 ],
        [-3.655899  , -2.6283965 , -2.105835  , ..., -0.3194244 ,
         -0.20551443, -0.08163142],
        [-1.5578523 , -2.2340577 , -1.6373229 , ...,  1.0037262 ,
          1.07352   ,  1.1719604 ]],

       [[ 0.10015258,  0.2876831 ,  0.34819946, ..., -0.8394653 ,
         -1.4103577 , -1.1443787 ],
        [-0.09712523,  0.17060548,  0.30126342, ..., -1.0547669 ,
         -0.7211792 , -0.8593872 ],
        [-0.2712036 , -0.01098633,  0.17915039, ..., -0.33385623,
         -0.5325745 , -0.75307006],
        ...,
        [-0.5039611 , -0.7769103 , -1.7374482 , ..., -4.6907225 ,
         -4.813266  , -4.964099  ],
        [-3.0827727 , -2.437619  , -2.218039  , ..., -3.5815887 ,
         -3.7604249 , -3.9050355 ],
        [-1.2271454 , -1.5318635 , -0.94440603, ..., -3.4824097 ,
         -3.3054414 , -3.2247558 ]]], dtype=float32)
Coordinates:
  * lat        (lat) float32 524B 15.0 15.5 16.0 16.5 ... 78.5 79.0 79.5 80.0
  * lon        (lon) float32 964B -80.0 -79.5 -79.0 -78.5 ... 39.0 39.5 40.0
  * time       (time) datetime64[ns] 15kB 1990-01-01 1990-01-02 ... 1994-12-31
    dayofyear  (time) int64 15kB 1 2 3 4 5 6 7 8 ... 359 360 361 362 363 364 365

In [65]:
df = da.to_dataframe().reset_index(allow_duplicates=True)
df = pl.from_pandas(df)

In [66]:
df

time,lat,lon,t
datetime[ns],f32,f32,f32
1990-01-01 00:00:00,15.0,-80.0,-0.034454
1990-01-01 00:00:00,15.0,-79.5,-0.090149
1990-01-01 00:00:00,15.0,-79.0,-0.070496
1990-01-01 00:00:00,15.0,-78.5,-0.05545
1990-01-01 00:00:00,15.0,-78.0,-0.111481
…,…,…,…
1994-12-31 00:00:00,80.0,38.0,-2.764069
1994-12-31 00:00:00,80.0,38.5,-2.632217
1994-12-31 00:00:00,80.0,39.0,-2.457748


In [ ]:
df.group_by(["lon", "lat", pl.col("time").dt.ordinal_day().alias("dayofyear")], maintain_order=True).agg(
    pl.col("t").mean()
).filter(pl.col("dayofyear") < 366)

lon,lat,dayofyear,t
f32,f32,i16,f32
-80.0,15.0,1,-0.132019
-79.5,15.0,1,-0.137933
-79.0,15.0,1,-0.150079
-78.5,15.0,1,-0.144946
-78.0,15.0,1,-0.16684
…,…,…,…
38.0,80.0,365,0.880615
38.5,80.0,365,0.981708
39.0,80.0,365,1.024661


In [53]:
clim = pl.col("t").mean()
anom = pl.col("t") - clim
dayofyear = pl.col("time").dt.ordinal_day().alias("dayofyear")
(
    df
    .group_by(["lon", "lat", dayofyear])
    .agg(clim=clim, anom=anom, time=pl.col("time"))
).explode("time", "anom")

lon,lat,dayofyear,clim,anom,time
f32,f32,i16,f32,f32,datetime[ns]
18.0,50.5,317,-2.149432,0.951312,1990-11-13 00:00:00
18.0,50.5,317,-2.149432,3.726062,1991-11-13 00:00:00
18.0,50.5,317,-2.149432,3.102405,1992-11-12 00:00:00
18.0,50.5,317,-2.149432,-1.514233,1993-11-13 00:00:00
18.0,50.5,317,-2.149432,-6.265546,1994-11-13 00:00:00
…,…,…,…,…,…
-59.0,38.5,125,0.017297,-2.901422,1990-05-05 00:00:00
-59.0,38.5,125,0.017297,-2.077447,1991-05-05 00:00:00
-59.0,38.5,125,0.017297,2.01853,1992-05-04 00:00:00
